<a href="https://colab.research.google.com/github/kabinet557/ksp2026/blob/abhineetavhad/XRB_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install astroquery astropy requests pandas openpyxl tqdm -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 99.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 55.6 MB/s eta 0:00:00


In [ ]:
from astroquery.heasarc import Heasarc
import pandas as pd

print("🔍 1. FETCHING ALL LIVE MISSION CATALOGS FROM HEASARC...")
try:
    # Use astroquery's native catalog lister instead of manual SQL
    cats_table = Heasarc.list_catalogs()
    cats_df = cats_table.to_pandas()
    cats_df.columns = cats_df.columns.str.lower()

    # Filter catalogs matching XMM, Chandra/CSC, or Swift
    matches = cats_df[cats_df['name'].str.contains('xmm|csc|swift|chandra', case=False, na=False)]

    print(f" Found {len(matches)} matching tables. Here are the catalog names to use:")
    print("-" * 80)
    print(matches[['name', 'description']].to_string(index=False))
    print("-" * 80)

except Exception as e:
    print(f"⚠️ Failed to pull catalog list: {e}")


print("\n🔍 2. INSPECTING AVAILABLE FLUX/RATE COLUMNS FOR SWIFT...")
# Let's test both potential internal shortnames for the Swift table
for swift_table_name in ['swift2sxps', 'swift2sxpscat']:
    try:
        cols_table = Heasarc.list_columns(swift_table_name)
        if cols_table and len(cols_table) > 0:
            cols_df = cols_table.to_pandas()
            cols_df.columns = cols_df.columns.str.lower()

            # Filter columns containing flux or rate
            flux_cols = cols_df[cols_df['name'].str.contains('flux|rate', case=False, na=False)]

            print(f"\n Found columns for table '{swift_table_name}':")
            print("=" * 60)
            print(flux_cols[['name', 'description']].to_string(index=False))
            print("=" * 60)
            break
    except:
        continue
else:
    print("⚠️ Could not retrieve column map for Swift tables via default handles.")

🔍 1. FETCHING ALL LIVE MISSION CATALOGS FROM HEASARC...
 Found 106 matching tables. Here are the catalog names to use:
--------------------------------------------------------------------------------
      name                                                      description
  a2lcscan                                    HEAO 1 A2 Scanned Lightcurves
agnsdssxmm Sloan Digital Sky Survey/XMM-Newton AGN Spectral Properties Cata
 alfperxmm           Alpha Per Open Cluster XMM-Newton X-Ray Source Catalog
atlascscpt AT Large Area Survey (ATLAS) CDF-S/SWIRE 1.4-GHz Components Cata
 carinaxmm     Carina OB1 Association XMM-Newton X-Ray Point Source Catalog
   cepaxmm              Cepheus A SFR XMM-Newton X-Ray Point Source Catalog
cfhtlsgxmm                 XMM-Newton CFHTLS W1 Field Galaxy Groups Catalog
 cmaob1xmm                    CMa OB1 XMM-Newton X-Ray Point Source Catalog
 coll69xmm       Collinder 69 Cluster XMM-Newton X-Ray Point Source Catalog
       csc                              

In [ ]:
import time, requests, warnings
import numpy as np
import pandas as pd
from io import StringIO
from tqdm import tqdm
from astroquery.vizier import Vizier
from astroquery.heasarc import Heasarc
from google.colab import files
from astropy.coordinates import SkyCoord
import astropy.units as u

warnings.filterwarnings("ignore")
Vizier.ROW_LIMIT = -1
Vizier.TIMEOUT = 30

RADIUS_ARCSEC    = 5.0
SLEEP            = 0.5
CHECKPOINT_EVERY = 25
MAX_SOURCES      = None

ZTF_URL = "https://irsa.ipac.caltech.edu/cgi-bin/ZTF/nph_light_curves"


def tap_query(adql):
    """Executes ADQL against HEASARC using astroquery's built-in TAP handler."""
    try:
        res = Heasarc.query_tap(adql)
        return res.to_table().to_pandas()
    except Exception:
        return pd.DataFrame()


def load_sources():
    frames = []
    heasarc_cats = [
        ("hmxbcat2", "name", "ra", "dec", "HMXB_Fortin2023"),
        ("hmxbcat",  "name", "ra", "dec", "HMXB_Liu2006"),
        ("lmxbcat",  "name", "ra", "dec", "LMXB_Liu2007"),
    ]

    for table, nc, rc, dc, label in heasarc_cats:
        df = tap_query(f"SELECT {nc}, {rc}, {dc} FROM public.{table}")

        if df.empty:
            df = tap_query(f"SELECT * FROM public.{table}")
            if not df.empty:
                df.columns = df.columns.str.lower()

        if df.empty:
            print(f"{label}: no results from either TAP or fallback")
            continue

        df = df.rename(columns={nc: "name", rc: "ra", dc: "dec"})
        df["cat"] = label
        df = df[["name", "ra", "dec", "cat"]].dropna(subset=["ra", "dec"])
        frames.append(df)
        print(f"{label}: {len(df)} sources")

    try:
        pass
    except Exception as e:
        print(f"ART-XC_1yr failed: {e}")

    if not frames:
        raise RuntimeError("No catalogues loaded — check network access in Colab.")

    all_sources = pd.concat(frames, ignore_index=True)
    all_sources["ra"]  = pd.to_numeric(all_sources["ra"],  errors="coerce")
    all_sources["dec"] = pd.to_numeric(all_sources["dec"], errors="coerce")
    all_sources = all_sources.dropna(subset=["ra", "dec"]).reset_index(drop=True)

    coords = SkyCoord(ra=all_sources["ra"].values * u.deg,
                      dec=all_sources["dec"].values * u.deg)

    keep, used = [], np.zeros(len(all_sources), dtype=bool)
    for i in range(len(all_sources)):
        if used[i]: continue
        close = coords[i].separation(coords) < RADIUS_ARCSEC * u.arcsec
        used[close] = True
        keep.append(i)
        all_sources.loc[i, "cat"] = "|".join(
            all_sources.loc[close.nonzero()[0], "cat"].unique())

    result = all_sources.iloc[keep].reset_index(drop=True)
    print(f"\nTotal unique sources after deduplication: {len(result)}")
    return result


def query_ztf(ra, dec):
    out = {f"ztf_{k}_{b}": np.nan
           for k in ["n", "med_mag", "range_mag"] for b in "gri"}
    out.update({"ztf_matched": False, "ztf_jd_min": np.nan, "ztf_jd_max": np.nan})
    try:
        r = requests.get(ZTF_URL, params={
            "POS": f"CIRCLE {ra} {dec} {RADIUS_ARCSEC/3600:.6f}",
            "BANDNAME": "g,r,i", "BAD_CATFLAGS_MASK": 32768, "FORMAT": "ipac_table"
        }, timeout=30)
        lines = [l for l in r.text.splitlines() if not l.startswith("\\")]
        df = pd.read_csv(StringIO("\n".join(lines)), sep=r"\s*\|\s*",
                         engine="python", skiprows=1)
        df.columns = df.columns.str.strip()
        df = df.loc[:, df.columns != ""]
        if df.empty or "mag" not in df.columns: return out
        out["ztf_matched"] = True
        band_col = "filtercode" if "filtercode" in df.columns else "band"
        for band, label in [("zg", "g"), ("zr", "r"), ("zi", "i")]:
            sub = df[df[band_col].str.strip() == band]["mag"].dropna()
            if sub.empty: continue
            out[f"ztf_n_{label}"]         = len(sub)
            out[f"ztf_med_mag_{label}"]   = round(float(sub.median()), 3)
            out[f"ztf_range_mag_{label}"] = round(float(sub.max() - sub.min()), 3)
        if "hjd" in df.columns:
            out["ztf_jd_min"] = float(df["hjd"].min())
            out["ztf_jd_max"] = float(df["hjd"].max())
    except: pass
    return out


def query_xray(ra, dec):
    rad    = RADIUS_ARCSEC / 3600.0
    circle = f"CIRCLE('ICRS',{ra},{dec},{rad})"

    # Updated table mappings and exact column profiles discovered via metadata lookup
    archives = {
        "4xmm":  (f"SELECT ep_8_flux   FROM public.xmmssc      WHERE CONTAINS(POINT('ICRS',ra,dec),{circle})=1", "ep_8_flux"),
        "csc2":  (f"SELECT flux_aper_b FROM public.csc         WHERE CONTAINS(POINT('ICRS',ra,dec),{circle})=1", "flux_aper_b"),
        "swift": (f"SELECT pow_flux_a  FROM public.swift2sxps  WHERE CONTAINS(POINT('ICRS',ra,dec),{circle})=1", "pow_flux_a"),
    }
    out = {}
    for label, (adql, flux_col) in archives.items():
        df = tap_query(adql)
        detected = not df.empty and len(df) > 0
        out[f"{label}_detected"] = detected
        out[f"{label}_flux"]     = float(df[flux_col].max()) if detected and flux_col in df.columns else np.nan
    out["any_xray"] = any(out[f"{l}_detected"] for l in ["4xmm", "csc2", "swift"])
    return out


def run():
    sources = load_sources()
    if MAX_SOURCES:
        sources = sources.head(MAX_SOURCES)

    results = []
    for i, row in tqdm(sources.iterrows(), total=len(sources), desc="Searching"):
        entry = {"name": row["name"], "ra": row["ra"],
                 "dec": row["dec"],  "catalogues": row["cat"]}
        entry.update(query_ztf(row["ra"], row["dec"]))
        entry.update(query_xray(row["ra"], row["dec"]))
        results.append(entry)
        time.sleep(SLEEP)

        if (i + 1) % CHECKPOINT_EVERY == 0:
            fname = f"xrb_checkpoint_{i+1}.xlsx"
            pd.DataFrame(results).to_excel(fname, index=False)
            print(f"\n Checkpoint saved to environment — {fname}")
            try:
                files.download(fname)
            except: pass

    final = pd.DataFrame(results)
    final.to_excel("xrb_final.xlsx", index=False)
    print("\n Done! File generation complete.")
    try:
        files.download("xrb_final.xlsx")
    except: pass
    return final

df = run()

HMXB_Fortin2023: 164 sources
HMXB_Liu2006: 114 sources
LMXB_Liu2007: 187 sources

Total unique sources after deduplication: 398


Searching:   6%|▌         | 24/398 [04:55<1:11:00, 11.39s/it]


 Checkpoint saved to environment — xrb_checkpoint_25.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

Searching: 100%|██████████| 398/398 [1:32:44<00:00, 13.98s/it]


 Done! File generation complete.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import time, requests, warnings
import numpy as np
import pandas as pd
from tqdm import tqdm
from io import StringIO
from astropy.io import ascii
from google.colab import files
warnings.filterwarnings("ignore")
RADIUS_ARCSEC    = 5.0
SLEEP            = 0.5
CHECKPOINT_EVERY = 25

ZTF_URL = "https://irsa.ipac.caltech.edu/cgi-bin/ZTF/nph_light_curves"

def query_ztf(ra, dec):
    out = {f"ztf_{k}_{b}": np.nan for k in ["n", "med_mag", "range_mag"] for b in "gri"}
    out.update({"ztf_matched": False, "ztf_jd_min": np.nan, "ztf_jd_max": np.nan})

    try:
        radius_deg = RADIUS_ARCSEC / 3600.0
        circle_val = f"{ra} {dec} {radius_deg:.6f}"

        r = requests.get(ZTF_URL, params={
            "CIRCLE": circle_val,
            "BANDNAME": "g,r,i",
            "BAD_CATFLAGS_MASK": 32768,
            "FORMAT": "ipac_table"
        }, timeout=30)

        if not r.text or "mag" not in r.text.lower():
            return out
        tbl = ascii.read(r.text, format='ipac')
        df = tbl.to_pandas()

        if df.empty or "mag" not in df.columns:
            return out

        out["ztf_matched"] = True
        band_col = "filtercode" if "filtercode" in df.columns else "band"
        for band, label in [("zg", "g"), ("zr", "r"), ("zi", "i")]:
            df[band_col] = df[band_col].astype(str).str.strip()
            sub = df[df[band_col] == band]["mag"].dropna()
            if sub.empty: continue

            out[f"ztf_n_{label}"]         = len(sub)
            out[f"ztf_med_mag_{label}"]   = round(float(sub.median()), 3)
            out[f"ztf_range_mag_{label}"] = round(float(sub.max() - sub.min()), 3)
        for time_col in ['hjd', 'mjd', 'mjdobs']:
            if time_col in df.columns and not df[time_col].dropna().empty:
                out["ztf_jd_min"] = float(df[time_col].min())
                out["ztf_jd_max"] = float(df[time_col].max())
                break

    except Exception as e:
        pass
    return out
print("📥 Please upload your existing Excel file containing the X-ray data:")
uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No file was uploaded. Please run the cell again.")
filename = list(uploaded.keys())[0]
sources_df = pd.read_excel(filename)

sources_df.columns = sources_df.columns.str.lower()
if 'ra' not in sources_df.columns or 'dec' not in sources_df.columns:
    raise KeyError("Could not find 'ra' or 'dec' columns in your spreadsheet. Check your column names!")
print(f"\n📂 Successfully loaded '{filename}' with {len(sources_df)} sources.")
print("🚀 Launching optical light-curve matching extraction loop...\n")

results = []
for i, row in tqdm(sources_df.iterrows(), total=len(sources_df), desc="Fetching ZTF Data"):
    ztf_data = query_ztf(row["ra"], row["dec"])
    combined_entry = row.to_dict()
    combined_entry.update(ztf_data)
    results.append(combined_entry)

    time.sleep(SLEEP)
    if (i + 1) % CHECKPOINT_EVERY == 0:
        checkpoint_name = f"optical_checkpoint_{i+1}.xlsx"
        pd.DataFrame(results).to_excel(checkpoint_name, index=False)
        print(f"\n💾 Saved progress checkpoint — {checkpoint_name}")
        try:
            files.download(checkpoint_name)
        except:
            pass

final_df = pd.DataFrame(results)
output_filename = "xrb_plus_optical_final.xlsx"
final_df.to_excel(output_filename, index=False)

print(f"\n🎉 Done! All optical columns have been extracted and integrated.")
print(f"📥 Downloading final consolidated file: '{output_filename}'")
try:
    files.download(output_filename)
except Exception:
    print("Automatic download blocked by browser settings. You can find the file in your Colab files tab.")

📥 Please upload your existing Excel file containing the X-ray data:


Saving xrb_final.xlsx to xrb_final (1).xlsx

📂 Successfully loaded 'xrb_final (1).xlsx' with 398 sources.
🚀 Launching optical light-curve matching extraction loop...



Fetching ZTF Data:   6%|▌         | 24/398 [03:38<54:18,  8.71s/it]


💾 Saved progress checkpoint — optical_checkpoint_25.xlsx


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

Fetching ZTF Data: 100%|██████████| 398/398 [1:11:31<00:00, 10.78s/it]


🎉 Done! All optical columns have been extracted and integrated.
📥 Downloading final consolidated file: 'xrb_plus_optical_final.xlsx'


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>